In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# project root = current working directory (your setup)
PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


In [2]:
# sanity check – versions must load without error
import numpy as np
import pandas as pd
import faiss
import torch

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS:", faiss.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


NumPy: 1.26.4
Pandas: 2.3.3
FAISS: 1.13.2
Torch: 2.9.1+cpu
CUDA available: False


In [3]:

DATA_PATH = "data"

docs = pd.read_csv(f"{DATA_PATH}/rag_corpus_documents.csv")
chunks = pd.read_csv(f"{DATA_PATH}/rag_corpus_chunks.csv")
qa_runs = pd.read_csv(f"{DATA_PATH}/rag_qa_eval_runs.csv")
retrievals = pd.read_csv(f"{DATA_PATH}/rag_retrieval_events.csv")
scenarios = pd.read_csv(f"{DATA_PATH}/rag_qa_scenarios.csv")
dictionary = pd.read_csv(f"{DATA_PATH}/rag_qa_data_dictionary.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'data/rag_corpus_documents.csv'

In [4]:
from pathlib import Path
import pandas as pd

# project root = parent of notebook
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data"

docs = pd.read_csv(DATA_PATH / "rag_corpus_documents.csv")
chunks = pd.read_csv(DATA_PATH / "rag_corpus_chunks.csv")
qa_runs = pd.read_csv(DATA_PATH / "rag_qa_eval_runs.csv")
retrievals = pd.read_csv(DATA_PATH / "rag_retrieval_events.csv")
scenarios = pd.read_csv(DATA_PATH / "rag_qa_scenarios.csv")
dictionary = pd.read_csv(DATA_PATH / "rag_qa_data_dictionary.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Vohita\\RAG\\data\\rag_qa_eval_runs.csv'

In [5]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data"

docs = pd.read_csv(DATA_PATH / "rag_corpus_documents.csv")
chunks = pd.read_csv(DATA_PATH / "rag_corpus_chunks.csv")
qa_runs = pd.read_csv(DATA_PATH / "eval_runs.csv")   # ← FIX HERE
retrievals = pd.read_csv(DATA_PATH / "rag_retrieval_events.csv")
scenarios = pd.read_csv(DATA_PATH / "rag_qa_scenarios.csv")
dictionary = pd.read_csv(DATA_PATH / "rag_qa_data_dictionary.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Vohita\\RAG\\data\\rag_qa_scenarios.csv'

In [6]:
sorted(p.name for p in DATA_PATH.iterdir())


['.ipynb_checkpoints',
 'data_dictionary.csv',
 'eval_runs.csv',
 'rag_corpus_chunks.csv',
 'rag_corpus_documents.csv',
 'rag_retrieval_events.csv',
 'scenarios.csv']

In [1]:
from pathlib import Path
import pandas as pd

# project root
PROJECT_ROOT = Path.cwd().parent

# paths
GENERAL_DATA_PATH = PROJECT_ROOT / "data"
LEGAL_DATA_PATH = PROJECT_ROOT / "data_legal"

# -------- GENERAL RAG (UNCHANGED) --------
docs = pd.read_csv(GENERAL_DATA_PATH / "rag_corpus_documents.csv")
qa_runs = pd.read_csv(GENERAL_DATA_PATH / "eval_runs.csv")
retrievals = pd.read_csv(GENERAL_DATA_PATH / "rag_retrieval_events.csv")
scenarios = pd.read_csv(GENERAL_DATA_PATH / "scenarios.csv")
dictionary = pd.read_csv(GENERAL_DATA_PATH / "data_dictionary.csv")

# -------- LEGAL RAG (NEW) --------
chunks = pd.read_csv(LEGAL_DATA_PATH / "rag_corpus_chunks.csv")

print("Loaded data successfully")



Loaded data successfully


In [2]:
tables = {
    "documents": docs,
    "chunks": chunks,
    "qa_runs": qa_runs,
    "retrievals": retrievals,
    "scenarios": scenarios,
    "dictionary": dictionary
}

for name, df in tables.items():
    print(f"{name:15s} -> rows: {df.shape[0]:6d}, cols: {df.shape[1]}")

documents       -> rows:    658, cols: 19
chunks          -> rows:    458, cols: 6
qa_runs         -> rows:   3824, cols: 49
retrievals      -> rows:  93375, cols: 12
scenarios       -> rows:     62, cols: 13
dictionary      -> rows:     99, cols: 5


In [3]:
chunks.columns

Index(['chunk_id', 'doc_id', 'domain', 'chunk_index', 'estimated_tokens',
       'chunk_text'],
      dtype='object')

In [4]:
docs.columns

Index(['doc_id', 'domain', 'title', 'source_type', 'language', 'n_sections',
       'n_tokens', 'n_chunks', 'avg_chunk_tokens', 'created_at_utc',
       'last_updated_at_utc', 'is_active', 'contains_tables', 'pii_risk_level',
       'security_tier', 'embedding_model', 'owner_team', 'search_index',
       'top_keywords'],
      dtype='object')

In [5]:
# creating stable row id
chunks = chunks.reset_index(drop=True)    
chunks["row_id"] = chunks.index

In [6]:
# meta dta to chunks
chunk_w_metadata = chunks.merge(
    docs[["doc_id", "source_type", "language", "created_at_utc", "last_updated_at_utc", "is_active", "contains_tables",
          "pii_risk_level", "security_tier", "owner_team"]], on="doc_id", how="left")

In [7]:
chunk_w_metadata.head(3)

,chunk_id,doc_id,domain,chunk_index,estimated_tokens,chunk_text,row_id,source_type,language,created_at_utc,last_updated_at_utc,is_active,contains_tables,pii_risk_level,security_tier,owner_team
0,LC00001,LEGAL_DOC_001,legal,0,51,Company shall not specify the business practic...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,LC00002,LEGAL_DOC_001,legal,0,308,In the event that Licensor grants to another V...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LC00003,LEGAL_DOC_001,legal,0,43,During the License Term (which is identified i...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# Create stable row_id. Never change ordering after this
chunk_w_metadata = chunk_w_metadata.reset_index(drop=True)
chunk_w_metadata["row_id"] = chunk_w_metadata.index

In [8]:
chunk_w_metadata.head(3)

,chunk_id,doc_id,domain,chunk_index,estimated_tokens,chunk_text,row_id,source_type,language,created_at_utc,last_updated_at_utc,is_active,contains_tables,pii_risk_level,security_tier,owner_team
0,LC00001,LEGAL_DOC_001,legal,0,51,Company shall not specify the business practic...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,LC00002,LEGAL_DOC_001,legal,0,308,In the event that Licensor grants to another V...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LC00003,LEGAL_DOC_001,legal,0,43,During the License Term (which is identified i...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# # Security tier normalization
# SECURITY_TIER_MAP = {
#     "public": 0,
#     "internal": 1,
#     "restricted": 2,
#     "highly_restricted": 3,
# }

# chunk_w_metadata["security_tier_level"] = (
#     chunk_w_metadata["security_tier"]
#     .map(SECURITY_TIER_MAP)
# )

In [10]:
# chunk_w_metadata["security_tier_level"].isnull().sum()

458

In [11]:
chunk_w_metadata.head(3)

,chunk_id,doc_id,domain,chunk_index,estimated_tokens,chunk_text,row_id,source_type,language,created_at_utc,last_updated_at_utc,is_active,contains_tables,pii_risk_level,security_tier,owner_team,security_tier_level
0,LC00001,LEGAL_DOC_001,legal,0,51,Company shall not specify the business practic...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,LC00002,LEGAL_DOC_001,legal,0,308,In the event that Licensor grants to another V...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LC00003,LEGAL_DOC_001,legal,0,43,During the License Term (which is identified i...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
# from pathlib import Path

# # base data path (same logic you’re already using)
# METADATA_DIR = Path.cwd().parent / "data" / "metadata"

# # create directory if it doesn't exist
# METADATA_DIR.mkdir(parents=True, exist_ok=True)

# # persist metadata
# chunk_w_metadata[metadata_cols].to_csv(
#     METADATA_DIR / "chunk_metadata.csv",
#     index=False
# )


In [22]:
# METADATA_DIR.exists(), list(METADATA_DIR.iterdir())


(True, [WindowsPath('C:/Users/Vohita/RAG/data/metadata/chunk_metadata.csv')])

In [23]:
# # Persist metadata store
# metadata_cols = [
#     "row_id",
#     "chunk_id",
#     "security_tier_level",
#     "owner_team",
#     "domain"
# ]

# chunk_w_metadata[metadata_cols].to_csv(
#     "../data/metadata/chunk_metadata.csv",
#     index=False
# )